In [ ]:
import pandas as pd
import numpy as np
import copy
import ast
import re 
from pyreadstat import read_sav, metadata_container
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.utils.validation import check_is_fitted
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from factor_analyzer.factor_analyzer import FactorAnalyzer
from factor_analyzer.factor_analyzer import calculate_kmo
from factor_analyzer import (ConfirmatoryFactorAnalyzer, ModelSpecificationParser)
from pathlib import Path

def transform_with_captured_group(text, capturing_group:str=None, prefix:str="", suffix:str="")->str:
    """
    Extract the first captured group from text and optionally add a prefix
    and/or suffix.

    Parameters
    ----------
    text : str
        Text to search.
    capturing_group : str, optional
        Regular expression containing a capturing group. The first match's
        first capturing group is returned.
    prefix : str, default=""
        Text to prepend to the captured group.
    suffix : str, default=""
        Text to append to the captured group.

    Returns
    -------
    str
        The prefix, captured text, and suffix concatenated.

    Raises
    ------
    AttributeError
        If the pattern does not match or has no capturing group.
    """
    return f"{prefix}{re.search(capturing_group, text).group(1)}{suffix}"

def same_answer_quality_check(dataframe_to_update:pd.DataFrame, column_selection_like:str="", column_selection_regex:str="") -> pd.DataFrame:
    """
    Function that checks strightlining behaviour or same anwer patterns for a series of variables. Useful to detecting quality issues with the data set used in segmentations. 
    
    :param dataframe_to_update: Original Data Frame
    :type dataframe_to_update: pd.DataFrame
    :param column_selection_like: Selecting the columns to analyze with a pd.DataFrame.filter(like="Pattern") function. 
    :type column_selection_like: str
    :param column_selection_regex: Selecting the columns to analyze with a pd.DataFrame.filter(regex=r"Pattern") function. 
    :type column_selection_regex: str
    :return: Dataframe with the added columns of data quality. Uses the filter "like" or "regex" patter as the column name with "_QUALITY_SCORE" suffix as the name. 
    :rtype: DataFrame
    """
    
    updated_frame = dataframe_to_update.copy()

    if column_selection_like != "":
        mode_series = updated_frame.filter(like=column_selection_like).mode(axis=1)[0]
        equal_counts_series = updated_frame.filter(like=column_selection_like).eq(mode_series, axis="index").sum(axis=1)
        total_counts_series = updated_frame.filter(like=column_selection_like).notna().sum(axis=1)
        updated_frame[f"{column_selection_like}_QUALITY_SCORE"] = equal_counts_series.div(total_counts_series)

    elif column_selection_regex != "":
        mode_series = updated_frame.filter(regex=column_selection_regex).mode(axis=1)[0]
        equal_counts_series = updated_frame.filter(regex=column_selection_regex).eq(mode_series, axis="index").sum(axis=1)
        total_counts_series = updated_frame.filter(regex=column_selection_regex).notna().sum(axis=1)
        updated_frame[f"{column_selection_regex}_QUALITY_SCORE"] = equal_counts_series.div(total_counts_series)
    else:
        print("ERROR: Please supply a filter valid filter string, either as like filter or regex filter.")
    
    return updated_frame

def rename_meta_variables(meta:metadata_container, rename_map: dict, inplace: bool = False)-> metadata_container:
    """
    Return a metadata_container whose variable-name-keyed structures use the new names.

    Parameters
    ----------
    meta : pyreadstat metadata_container
    rename_map : dict  {old_variable_name: new_variable_name}
    inplace : bool     modify the given meta instead of a copy
    """
    new_meta = meta if inplace else copy.deepcopy(meta)

    def _rn(name):
        return rename_map.get(name, name)

    # 1) attributes that are dicts keyed by variable name
    dict_attrs = [
        "column_names_to_labels",
        "variable_value_labels",
        "variable_to_label",
        "variable_measure",
        "variable_display_width",
        "variable_alignment",
        "variable_storage_width",
        "original_variable_types",
        "readstat_variable_types",
        "missing_ranges",
        "missing_user_values",
        "variable_format",
    ]
    for attr in dict_attrs:
        old = getattr(new_meta, attr, None)
        if isinstance(old, dict):
            setattr(new_meta, attr, {_rn(k): v for k, v in old.items()})

    # 2) attributes that are lists of variable names
    list_attrs = ["column_names", "mr_sets"]
    for attr in list_attrs:
        old = getattr(new_meta, attr, None)
        if isinstance(old, list):
            setattr(new_meta, attr, [_rn(v) for v in old])

    # 3) multiple response sets: dict of set_name -> {..., "variable_list": [...]}
    mrsets = getattr(new_meta, "mrset", None) or getattr(new_meta, "multiple_response_sets", None)
    if isinstance(mrsets, dict):
        for set_name, cfg in mrsets.items():
            if isinstance(cfg, dict) and isinstance(cfg.get("variable_list"), list):
                cfg["variable_list"] = [_rn(v) for v in cfg["variable_list"]]

    return new_meta

def dataframe_stacking(df:pd.DataFrame, id_vars_list:list, key_pattern:str, keep_key:bool=True, key_column_name:str="STACK_KEY") -> pd.DataFrame:
    """
    Reshape a DataFrame from wide format to stacked format.

    All columns not listed in ``id_vars_list`` are treated as value columns.
    Their names are transformed using ``key_pattern``. If ``keep_key`` is
    ``True``, the full regex match is stored in a key column; the matching
    text is removed from the column names in either case. Rows with missing
    values are dropped before the result is pivoted back to wide format.

    Parameters
    ----------
    df : pandas.DataFrame
        DataFrame to reshape.
    id_vars_list : list of str
        Columns to retain as identifier variables.
    key_pattern : str
        Regular expression matched against value-column names. The full match
        is used as the key when ``keep_key`` is ``True``.
    keep_key : bool, default=True
        Whether to include the extracted key as an identifier column.
    key_column_name : str, default="STACK_KEY"
        Name of the key column when ``keep_key`` is ``True``.

    Returns
    -------
    pandas.DataFrame
        Reshaped DataFrame with the identifier columns and pivoted values.
    """
    
    stacked_df = df.copy()
    stacked_df = stacked_df.melt(id_vars=id_vars_list)    
    extended_id_vars_list = id_vars_list.copy()
    extended_id_vars_list.append(key_column_name)

    if keep_key:
        stacked_df[key_column_name] = stacked_df['variable'].apply(lambda var: re.match(pattern=key_pattern, string=var)[0])
        
    stacked_df["variable"] = stacked_df["variable"].apply(lambda var: re.sub(pattern=key_pattern, repl="", string=var))
    stacked_df.dropna(subset=["value"], inplace=True)
    
    if keep_key:
        stacked_df = stacked_df.pivot(columns="variable", values="value", index=extended_id_vars_list)
    else: 
        stacked_df = stacked_df.pivot(columns="variable", values="value", index=id_vars_list)
    
    return stacked_df.reset_index()

def parse_stack_column_config(column_name: str) -> dict:
    """
    Parse a stack configuration column name into its embedded Python dictionary.

    The function expects column names produced by the stacking workflow, where
    the configuration is stored as a literal Python object after the
    "STACK_" prefix. For example, a column named
    "STACK_{'source': 'A', 'key': 'Q1'}" is converted back into the
    corresponding dictionary.

    Parameters
    ----------
    column_name : str
        Name of the stack configuration column. It must begin with "STACK_".

    Returns
    -------
    dict
        The dictionary represented by the text that follows the "STACK_" prefix.

    Raises
    ------
    ValueError
        If the column name does not start with the required "STACK_" prefix.
    """
    prefix = "STACK_"

    if not column_name.startswith(prefix):
        raise ValueError(f"Not a stack configuration column: {column_name}")

    config_str = column_name.removeprefix(prefix)

    return ast.literal_eval(config_str)

class ColumnMapper(BaseEstimator, TransformerMixin):

    def __init__(self, mappings):
        self.mappings = mappings

    def fit(self, X, y=None):
        self.feature_names_in_ = list(X.columns)
        return self

    def transform(self, X):
        X = X.copy()

        for col, mapping in self.mappings.items():
            if col in X.columns:
                X[col] = X[col].replace(mapping)

        return X

    def get_feature_names_out(self, input_features=None):
        return self.feature_names_in_
    
class TopNBoxEncoder(BaseEstimator, TransformerMixin):
    """
    Create cumulative Top-N-Box or Bottom-N-Box binary columns.

    Parameters
    ----------
    n_boxes : dict
        Column names mapped to the number of box columns to create.
        For example, {"A": 2} creates A_T1B and A_T2B when top=True,
        or A_B1B and A_B2B when top=False.

    limit_values : dict or None, default=None
        Theoretical maximum for each column when top=True, or minimum
        when top=False. Unspecified limits are learned from training data.

    handle_missing : {"zero", "nan"}, default="zero"
        Whether missing values produce zeros or remain missing.

    keep_original : bool, default=False
        Whether to include the original columns in the output.

    top : bool, default=True
        If True, encode top boxes; otherwise, encode bottom boxes.
    """

    def __init__(
        self,
        n_boxes,
        limit_values=None,
        handle_missing="zero",
        keep_original=False,
        top=True,
    ):
        self.n_boxes = n_boxes
        self.limit_values = limit_values
        self.handle_missing = handle_missing
        self.keep_original = keep_original
        self.top = top

    def fit(self, X, y=None):
        X = self._to_dataframe(X)

        self.feature_names_in_ = np.asarray(X.columns, dtype=object)
        self.n_features_in_ = X.shape[1]

        self._validate_parameters(X)

        # Store the upper or lower limit used for each column.
        self.limit_values_ = {}

        supplied_limit_values = (
            {} if self.limit_values is None else self.limit_values
        )

        for column in self.n_boxes:
            if column in supplied_limit_values:
                limit = supplied_limit_values[column]
            else:
                limit = (
                    X[column].max(skipna=True)
                    if self.top
                    else X[column].min(skipna=True)
                )

            if pd.isna(limit):
                raise ValueError(
                    f"Cannot determine the limit for column {column!r}. "
                    "The column contains only missing values. Provide its "
                    "limit through limit_values."
                )

            self.limit_values_[column] = limit

        self.output_feature_names_ = self._build_feature_names()

        return self

    def transform(self, X):
        check_is_fitted(
            self,
            attributes=[
                "feature_names_in_",
                "limit_values_",
                "output_feature_names_",
            ],
        )

        X = self._to_dataframe(
            X,
            expected_columns=self.feature_names_in_,
        )

        generated_columns = {}
        box_prefix = "T" if self.top else "B"

        if self.keep_original:
            for column in X.columns:
                generated_columns[column] = X[column].copy()

        for column, n_boxes in self.n_boxes.items():
            values = pd.to_numeric(X[column], errors="coerce")
            limit = self.limit_values_[column]

            for n in range(1, n_boxes + 1):
                output_name = f"{column}_{box_prefix}{n}B"

                if self.top:
                    threshold = limit - n + 1
                    encoded = values.ge(threshold) & values.le(limit)
                else:
                    threshold = limit + n - 1
                    encoded = values.le(threshold) & values.ge(limit)

                if self.handle_missing == "nan":
                    encoded = encoded.astype("Float64")
                    encoded = encoded.mask(values.isna(), pd.NA)
                else:
                    encoded = encoded.fillna(False).astype(np.uint8)

                generated_columns[output_name] = encoded

        return pd.DataFrame(
            generated_columns,
            index=X.index,
        )

    def get_feature_names_out(self, input_features=None):
        check_is_fitted(self, attributes=["output_feature_names_"])
        return np.asarray(self.output_feature_names_, dtype=object)

    def _build_feature_names(self):
        names = []
        box_prefix = "T" if self.top else "B"

        if self.keep_original:
            names.extend(self.feature_names_in_)

        for column, n_boxes in self.n_boxes.items():
            names.extend(
                f"{column}_{box_prefix}{n}B"
                for n in range(1, n_boxes + 1)
            )

        return np.asarray(names, dtype=object)

    def _validate_parameters(self, X):
        if not isinstance(self.n_boxes, dict):
            raise TypeError("n_boxes must be a dictionary.")

        if self.limit_values is not None:
            if not isinstance(self.limit_values, dict):
                raise TypeError("limit_values must be a dictionary or None.")

        if not isinstance(self.top, (bool, np.bool_)):
            raise TypeError("top must be a boolean.")

        if self.handle_missing not in {"zero", "nan"}:
            raise ValueError(
                "handle_missing must be either 'zero' or 'nan'."
            )

        missing_columns = set(self.n_boxes) - set(X.columns)

        if missing_columns:
            raise ValueError(
                "The following columns were not found in X: "
                f"{sorted(missing_columns)}"
            )

        invalid_n_values = {
            column: n
            for column, n in self.n_boxes.items()
            if (
                not isinstance(n, (int, np.integer))
                or isinstance(n, bool)
                or n < 1
            )
        }

        if invalid_n_values:
            raise ValueError(
                "The number of box variables must be a positive "
                f"integer. Invalid values: {invalid_n_values}"
            )

        supplied_limit_values = (
            {} if self.limit_values is None else self.limit_values
        )

        # Retain the original upper-limit check for top boxes only.
        if self.top:
            invalid_boxes = {
                column: n
                for column, n in self.n_boxes.items()
                if (
                    column in supplied_limit_values
                    and n > supplied_limit_values[column]
                )
            }

            if invalid_boxes:
                raise ValueError(
                    "The requested number of boxes cannot be greater than "
                    "the supplied maximum value. Invalid values: "
                    f"{invalid_boxes}"
                )

    @staticmethod
    def _to_dataframe(X, expected_columns=None):
        if isinstance(X, pd.DataFrame):
            X_df = X.copy()

            if expected_columns is not None:
                missing = set(expected_columns) - set(X_df.columns)

                if missing:
                    raise ValueError(
                        f"Missing columns during transform: {sorted(missing)}"
                    )

                X_df = X_df.loc[:, expected_columns]

            return X_df

        X_array = np.asarray(X)

        if X_array.ndim != 2:
            raise ValueError("X must be a two-dimensional structure.")

        if expected_columns is None:
            columns = [
                f"x{i}" for i in range(X_array.shape[1])
            ]
        else:
            if X_array.shape[1] != len(expected_columns):
                raise ValueError(
                    "The number of columns during transform does not match "
                    "the number seen during fit."
                )

            columns = expected_columns

        return pd.DataFrame(X_array, columns=columns)

def loadings_to_frame(model:FactorAnalyzer, data_frame:pd.DataFrame, labels_dict:dict)->pd.DataFrame:
    """
    Convert a fitted factor analyzer's loadings into a labeled DataFrame.

    Rows correspond to the input attributes, with their index labels mapped
    through `labels_dict`; columns correspond to the extracted factors.

    Parameters
    ----------
    model : FactorAnalyzer
        Fitted factor analyzer containing `loadings_`.
    data_frame : pandas.DataFrame
        Input data whose columns identify the attributes used to fit the model.
    labels_dict : dict
        Mapping from attribute column names to display labels.

    Returns
    -------
    pandas.DataFrame
        Factor loadings indexed by the mapped attribute labels.
    """
    output = pd.DataFrame(model.loadings_).T
    output.columns = data_frame.columns
    output = output.T
    output.index = output.index.map(labels_dict)
    return output

def merge_factors_to_data(model:FactorAnalyzer, factor_input_df:pd.DataFrame, original_df:pd.DataFrame, sample_label:str, grid_label:str, rotation_label:str, no_of_factors:int)->pd.DataFrame:
    """
    Transform input observations into factor scores and append them to a copy of
    the original data.

    Missing values in `factor_input_df` are filled with the corresponding
    column means before transformation. The resulting factor scores are named
    using the sample, grid, factor-count, rotation, and factor number, then
    joined to `original_df` by index with a left join.

    Parameters
    ----------
    model : FactorAnalyzer
        Fitted factor analyzer used to calculate factor scores.
    factor_input_df : pandas.DataFrame
        Input attributes to transform. Its index is used to align scores with
        `original_df`.
    original_df : pandas.DataFrame
        DataFrame to which the factor scores are added.
    sample_label : str
        Sample identifier used in the score column names.
    grid_label : str
        Grid identifier used in the score column names.
    rotation_label : str
        Rotation method used in the score column names.
    no_of_factors : int
        Number of factors used to fit `model`.

    Returns
    -------
    pandas.DataFrame
        A new DataFrame containing `original_df` and the appended factor-score
        columns.
    """
    output = pd.DataFrame(model.transform(factor_input_df.fillna(factor_input_df.mean())))
    output.index = factor_input_df.index
    output.columns = [f'{sample_label}_{grid_label}_{no_of_factors}F_{rotation_label}_{n}' for n in range(output.shape[1])]
    original_df = pd.merge(
        left=original_df,
        right=output,
        left_index=True,
        right_index=True,
        how='left'
    )

    return original_df

def ladder_sort(df:pd.DataFrame)->pd.DataFrame:
    """
    Group rows by the column containing each row's maximum value, then sort
    each group in descending order and concatenate the groups.

    Parameters
    ----------
    df : pandas.DataFrame
        DataFrame with numeric values to sort. Each row is assigned to every
        column in which it has a maximum value; ties may therefore cause a row
        to appear more than once in the result.

    Returns
    -------
    pandas.DataFrame
        A new DataFrame containing the grouped and sorted rows.
    """
    working_df = df.copy() # create a copy of input df
    max_values = working_df.max(axis=1) # Get row max 
    ordered_blocks = [] # Initialize sorted objects collector 

    for col in working_df.columns.to_list():
        block = working_df[working_df[col] == max_values] # Filter by the top values of each column 
        ordered_blocks.append(block.sort_values(by = col, ascending = False)) # Sort it and append it 

    return pd.concat(ordered_blocks) # Return the concatenated df

def factor_analysis_generation(data_frame:pd.DataFrame, sample_filters:dict, grids:dict, att_labels:dict ):
    """
    Run factor analyses across sample, attribute-grid, rotation, and factor-count combinations.

    For each combination, this function fits a principal-factor model. Models
    with a cumulative variance of at least 0.6 have their labeled loadings
    recorded and their factor scores appended to a copy of the input data.
    Fit exceptions are stored in the variance results. KMO scores are also
    calculated for each sample and grid.

    Parameters
    ----------
    data_frame : pandas.DataFrame
        Source data containing the sample and attribute columns.
    sample_filters : dict
        Mapping of sample names to Boolean masks selecting rows in
        `data_frame`.
    grids : dict
        Mapping of grid names to lists of attribute column names.
    att_labels : dict
        Mapping of attribute column names to display labels for loadings.

    Returns
    -------
    tuple
        A tuple containing:
        - `loading_matrices` : dict
          Maps keys of the form
          `"{sample}_{grid}_{factor_count}F_{rotation}"` to sorted,
          three-decimal-place loading DataFrames. Includes only models whose
          cumulative variance is at least 0.6.
        - `accumulated_variances` : dict
          Uses the same key format as `loading_matrices`. Values are the
          models' final cumulative variance values, or the exception raised
          when fitting failed.
        - `updated_data` : pandas.DataFrame
          A copy of `data_frame` with qualifying factor scores appended.
          Score columns are named
          `"{sample}_{grid}_{factor_count}F_{rotation}_{factor_number}"`.
        - `grid_KMO_diagnostics` : dict
          Maps keys of the form `"{sample}_{grid}_KMO Score"` to the
          overall KMO score for that sample and grid.
        - `attributes_KMO_diagnostics` : dict
          Maps attribute column names to their attribute-level KMO scores.
          If an attribute occurs in multiple sample/grid combinations, its
          value is overwritten by the last score calculated for it.
    """
    loading_matrices = {}
    accumulated_variances = {}
    updated_data = data_frame.copy()
    grid_KMO_diagnostics = {}
    attributes_KMO_diagnostics = {}

    rotations = ['promax','varimax','equamax']
    number_of_factors = np.arange(2,100)

    # Factor analysis main loop 
    for sample in sample_filters.keys():
        for grid in grids.keys():
            for rotation in rotations:
                for no in number_of_factors:

                    # If the iteration number is greater or equal to the number of attributes it skips this iteraiton.
                    if no >= len(grids[grid]):
                        continue
                    
                    # Filters the sample using the filter mask defined in the sample filters dictionary and selects the attribute columns.
                    filtered_df = data_frame[sample_filters[sample]][grids[grid]]

                    fa = FactorAnalyzer(n_factors=no, rotation=rotation, method='principal', impute='mean')

                    try:
                        fa.fit(filtered_df)
                    except Exception as e:
                        accumulated_variances[f'{sample}_{grid}_{no}F_{rotation}'] = e # Logs the error if there was an issue while fitting the model 
                        continue

                    accumulated_variances[f'{sample}_{grid}_{no}F_{rotation}'] = fa.get_factor_variance()[2][-1] # Logs the accumulated variance for the model

                    if fa.get_factor_variance()[2][-1] >= 0.6:
                        loadings_matrix = loadings_to_frame(
                            model=fa,
                            data_frame=filtered_df,
                            labels_dict=att_labels
                        )

                        loading_matrices[f'{sample}_{grid}_{no}F_{rotation}'] = ladder_sort(loadings_matrix).round(3)

                        updated_data = merge_factors_to_data(
                            model=fa, 
                            factor_input_df=filtered_df, 
                            original_df=updated_data, 
                            sample_label=sample, 
                            grid_label=grid, 
                            rotation_label=rotation, 
                            no_of_factors=no
                            )
                        
    for sample in sample_filters.keys():
        for grid in grids.keys():
            filtered_df = data_frame[sample_filters[sample]][grids[grid]]

            # Calculate KMO scores
            attribute_scores, grid_score = calculate_kmo(filtered_df)

            grid_KMO_diagnostics[f'{sample}_{grid}_KMO Score'] =  grid_score

            attributes_KMO_diagnostics.update(dict(zip(filtered_df.columns, attribute_scores)))
            

    
    return loading_matrices, accumulated_variances, updated_data, grid_KMO_diagnostics, attributes_KMO_diagnostics

# Step 1: Preprocessing and cleaning of dataset

In [ ]:
#### RUN ONLY FOR THE FIRST TIME! 

# Load database 
data_cleaning, meta_cleaning = read_sav("input_database.sav")

# VAR_CLEANING_FRAME preparation 
VAR_CLEANING_FRAME = data_cleaning.columns.to_frame(index=False, name="VAR")
VAR_CLEANING_FRAME[["N_VAR"]] = ""

# Prepare data cleaning Excel 
with pd.ExcelWriter("dataframe_configuration.xlsx", engine="openpyxl") as writer:
    VAR_CLEANING_FRAME.to_excel(writer, sheet_name="VAR_CLEANING_FRAME", index=False)

In [ ]:
#### RESUME WORK FROM HERE! 

# Loads the data cleaning configuration file. We fill the columns as following: 
# VAR = Original variable name, only keep those we are going to use
# N_VAR = New variable name. Copy original ones and update those needed for preprocessing reasons. 

data_cleaning, meta_cleaning = read_sav("input_database.sav")

VAR_CLEANING_FRAME = pd.read_excel("dataframe_configuration.xlsx", sheet_name="VAR_CLEANING_FRAME").set_index("VAR")

MAIN_VAR_DICT = VAR_CLEANING_FRAME["N_VAR"].to_dict()

# Loads, select variables and renames relevant columns
data_cleaning, meta_cleaning = read_sav("input_database.sav")
data_cleaning = data_cleaning[VAR_CLEANING_FRAME.index] # Selects only relevant variables 
meta_cleaning = rename_meta_variables(meta_cleaning, MAIN_VAR_DICT) # Renames meta
data_cleaning = data_cleaning.rename(columns=MAIN_VAR_DICT) # Renames column names

In [ ]:
#### RUN ONLY FOR THE FIRST TIME! 

# PIPE CONFIG FRAME preparation 
STACK_CONFIG_FRAME = data_cleaning.columns.to_frame(index=False, name="VAR")
STACK_CONFIG_FRAME[["ex_STACK_{\"key_pattern\": \"regex_pattern\", \"keep_key\": True, \"key_column_name\": \"STACK_KEY\"}"]] = ""

with pd.ExcelWriter("dataframe_configuration.xlsx", engine="openpyxl") as writer:
    VAR_CLEANING_FRAME.reset_index().to_excel(writer, sheet_name="VAR_CLEANING_FRAME", index=False)
    STACK_CONFIG_FRAME.to_excel(writer, sheet_name="STACK_CONFIG_FRAME", index=False)

In [ ]:
#### RESUME WORK FROM HERE! 

VAR_CLEANING_FRAME = pd.read_excel("dataframe_configuration.xlsx", sheet_name="VAR_CLEANING_FRAME").set_index("VAR")
STACK_CONFIG_FRAME = pd.read_excel("dataframe_configuration.xlsx", sheet_name="STACK_CONFIG_FRAME").set_index("VAR")

# Stacking loop
for col in STACK_CONFIG_FRAME.columns:

    if col.startswith("STACK_"):

        # print(f"{col}")

        # Parse parameters from column name
        params = parse_stack_column_config(col)

        # Extracting ID vars lists 
        ID_VARS_LIST_ = STACK_CONFIG_FRAME[STACK_CONFIG_FRAME[col]=="ID"].index.to_list()

        data_cleaning = dataframe_stacking(
            df=data_cleaning,
            id_vars_list=ID_VARS_LIST_,
            **params,
        )

        # print("Success!")

In [ ]:
#### RUN ONLY FOR THE FIRST TIME! 

# PIPE CONFIG FRAME preparation 
PIPE_CONFIG_FRAME = data_cleaning.columns.to_frame(index=False, name="VAR")
PIPE_CONFIG_FRAME[["DICT_REP","ONE_HOT","TOP_N","TOP_N_MAX", "BOTTOM_N", "BOTTOM_N_MIN"]] = ""

with pd.ExcelWriter("dataframe_configuration.xlsx", engine="openpyxl") as writer:
    VAR_CLEANING_FRAME.reset_index().to_excel(writer, sheet_name="VAR_CLEANING_FRAME", index=False)
    STACK_CONFIG_FRAME.reset_index().to_excel(writer, sheet_name="STACK_CONFIG_FRAME", index=False)
    PIPE_CONFIG_FRAME.to_excel(writer, sheet_name="PIPE_CONFIG_FRAME", index=False)

In [ ]:
#### RESUME WORK FROM HERE! 
VAR_CLEANING_FRAME = pd.read_excel("dataframe_configuration.xlsx", sheet_name="VAR_CLEANING_FRAME").set_index("VAR")
STACK_CONFIG_FRAME = pd.read_excel("dataframe_configuration.xlsx", sheet_name="STACK_CONFIG_FRAME").set_index("VAR")
PIPE_CONFIG_FRAME = pd.read_excel("dataframe_configuration.xlsx", sheet_name="PIPE_CONFIG_FRAME").set_index("VAR")


DICT_REP_COLS = PIPE_CONFIG_FRAME.dropna(subset="DICT_REP").index.to_list()
ONE_HOT_COLS = PIPE_CONFIG_FRAME.dropna(subset="ONE_HOT").index.to_list()
TOP_N_COLS = PIPE_CONFIG_FRAME.dropna(subset="TOP_N").index.to_list()
BOTTOM_N_COLS = PIPE_CONFIG_FRAME.dropna(subset="BOTTOM_N").index.to_list()

DICT_REP_MAPPING = PIPE_CONFIG_FRAME["DICT_REP"].dropna().apply(lambda x: ast.literal_eval(x)).to_dict()
TOP_N_COLS_MAPPING = PIPE_CONFIG_FRAME["TOP_N"].dropna().astype("int").to_dict()
TOP_N_COLS_MAX_MAPPING = PIPE_CONFIG_FRAME["TOP_N_MAX"].dropna().astype("int").to_dict()
BOTTOM_N_COLS_MAPPING = PIPE_CONFIG_FRAME["BOTTOM_N"].dropna().astype("int").to_dict()
BOTTOM_N_COLS_MIN_MAPPING = PIPE_CONFIG_FRAME["BOTTOM_N_MIN"].dropna().astype("int").to_dict()

# Instanciating preprocessing transformer 
preprocessing_trasnformer = ColumnTransformer([
    ("DRT", ColumnMapper(mappings=DICT_REP_MAPPING), DICT_REP_COLS),
    ("OHT", OneHotEncoder(sparse_output=False), ONE_HOT_COLS),
    ("TBT", TopNBoxEncoder(n_boxes=TOP_N_COLS_MAPPING, limit_values=TOP_N_COLS_MAX_MAPPING, handle_missing="zero", keep_original=True, top=True), TOP_N_COLS),
    ("BBT", TopNBoxEncoder(n_boxes=BOTTOM_N_COLS_MAPPING, limit_values=BOTTOM_N_COLS_MIN_MAPPING, handle_missing="zero", keep_original=True, top=False), BOTTOM_N_COLS),
], remainder="passthrough")

# Fit and transform 
preprocessing_trasnformer.fit(data_cleaning)
clean_data = pd.DataFrame(data=preprocessing_trasnformer.transform(data_cleaning), columns=preprocessing_trasnformer.get_feature_names_out())

output_dir = Path("./preprocessing_data")
output_dir.mkdir(parents=True, exist_ok=True)

clean_data.to_csv(output_dir / "clean_data.csv", index=False)
clean_data.to_pickle(output_dir / "clean_data.plk")

# Step 2: Preanalysis report and Factor Analysis

In [ ]:
#### RUN ONLY FOR THE FIRST TIME! 
clean_data = pd.read_pickle("preprocessing_data/clean_data.plk")

LABELS_CONFIG_FRAME =  clean_data.columns.to_frame(index=False, name="VAR")
LABELS_CONFIG_FRAME["LABEL"] = ""

PREANALISIS_CONFIG_FRAME = pd.DataFrame(index=["main_filter_mask", "subsample_filter_mask", "group_by_var"], columns=["analysis_label"])
PREANALISIS_CONFIG_FRAME.index.name = "SETTINGS"

FA_SAMPLE_CONFIG_FRAME = pd.DataFrame(index=["filter_mask"], columns=["fa_exercise_label"])
FA_SAMPLE_CONFIG_FRAME.index.name = "SETTINGS"

FA_CONFIG_FRAME =  clean_data.columns.to_frame(index=False, name="VAR")
FA_CONFIG_FRAME["fa_exercise_label"] =""

with pd.ExcelWriter("preanalysis_and_fa_configuration.xlsx", engine="openpyxl") as writer:
    LABELS_CONFIG_FRAME.to_excel(writer, sheet_name="LABELS_CONFIG_FRAME", index=False)
    PREANALISIS_CONFIG_FRAME.to_excel(writer, sheet_name="PREANALISIS_CONFIG_FRAME", index=True)
    FA_SAMPLE_CONFIG_FRAME.to_excel(writer, sheet_name="FA_SAMPLE_CONFIG_FRAME", index=True)
    FA_CONFIG_FRAME.to_excel(writer, sheet_name="FA_CONFIG_FRAME", index=False)

In [ ]:
#### RESUME WORK FROM HERE! 

clean_data = pd.read_pickle("preprocessing_data/clean_data.plk").infer_objects()

LABELS_CONFIG_FRAME = pd.read_excel("preanalysis_and_fa_configuration.xlsx", sheet_name="LABELS_CONFIG_FRAME", index_col="VAR")
PREANALISIS_CONFIG_FRAME = pd.read_excel("preanalysis_and_fa_configuration.xlsx", sheet_name="PREANALISIS_CONFIG_FRAME", index_col="SETTINGS")
FA_SAMPLE_CONFIG_FRAME = pd.read_excel("preanalysis_and_fa_configuration.xlsx", sheet_name="FA_SAMPLE_CONFIG_FRAME", index_col="SETTINGS")
FA_CONFIG_FRAME = pd.read_excel("preanalysis_and_fa_configuration.xlsx", sheet_name="FA_CONFIG_FRAME", index_col="VAR")

LABELS_DICT = LABELS_CONFIG_FRAME.to_dict()["LABEL"]
FA_SAMPLE_CONFIG_DICT = FA_SAMPLE_CONFIG_FRAME.astype(str).to_dict()

# Outputting the filtered datasets for ABSA / SmartPLS
output_dir_report = Path("./reports")
output_dir_report.mkdir(parents=True, exist_ok=True)

# PREANALYSIS LOOP 
preanalysis_report = {}

for config_col in PREANALISIS_CONFIG_FRAME.columns:

    config = PREANALISIS_CONFIG_FRAME[config_col].astype(str).to_dict()
    # print(config)
    
    filtered_frame = clean_data.copy()

    if config["main_filter_mask"] != "nan":

        # Applying main filter 
        filtered_frame = filtered_frame[filtered_frame.eval(config["main_filter_mask"])]
        

    # Preparing and adding labels to output frames 
    main_avg_frame = filtered_frame.mean(numeric_only=True).to_frame().copy()
    main_avg_frame.columns = ["mean"]
    main_avg_frame["Labels"] = main_avg_frame.index.map(LABELS_DICT)

    main_count_frame = filtered_frame.count().to_frame().copy()
    main_count_frame.columns = ["count"]
    main_count_frame["Labels"] = main_count_frame.index.map(LABELS_DICT)

    main_corr_frame = filtered_frame.corr(numeric_only=True).copy()
    main_corr_frame["Labels"] = main_corr_frame.index.map(LABELS_DICT)
    main_corr_frame = main_corr_frame.T
    main_corr_frame["Labels"] = main_corr_frame.index.map(LABELS_DICT)

    # Saving output frames to preanalysis_report dictionary
    preanalysis_report[f"{config_col}_avg"] = main_avg_frame
    preanalysis_report[f"{config_col}_count"] = main_count_frame
    preanalysis_report[f"{config_col}_corr"] = main_corr_frame    

    if config["subsample_filter_mask"] != "nan":

        # Filtering subsamples 
        subsample_filtered_frame = filtered_frame.copy()
        subsample_filtered_frame = subsample_filtered_frame[subsample_filtered_frame.eval(config["subsample_filter_mask"])]

        # Preparing and adding labels to output frames 
        ss_avg_frame = subsample_filtered_frame.mean(numeric_only=True).to_frame().copy()
        ss_avg_frame.columns = ["mean"]
        ss_avg_frame["Labels"] = ss_avg_frame.index.map(LABELS_DICT)

        ss_count_frame = subsample_filtered_frame.count().to_frame().copy()
        ss_count_frame.columns = ["count"]
        ss_count_frame["Labels"] = ss_count_frame.index.map(LABELS_DICT)

        ss_corr_frame = subsample_filtered_frame.corr(numeric_only=True).copy()
        ss_corr_frame["Labels"] = ss_corr_frame.index.map(LABELS_DICT)
        ss_corr_frame = ss_corr_frame.T
        ss_corr_frame["Labels"] = ss_corr_frame.index.map(LABELS_DICT)

        # Saving output frames into preanalysis_report dictionary
        preanalysis_report[f"{config_col}_ss_avg"] = ss_avg_frame
        preanalysis_report[f"{config_col}_ss_count"] = ss_count_frame
        preanalysis_report[f"{config_col}_ss_corr"] = ss_corr_frame

    if config["group_by_var"] != "nan":

        # Preparing and adding labels to output frames 
        xtab_frame = filtered_frame.groupby(by=config["group_by_var"]).mean(numeric_only=True).T.copy()
        xtab_frame["Labels"] = xtab_frame.index.map(LABELS_DICT)

        # Saving output frames into preanalysis_report dictionary
        preanalysis_report[f"{config_col}_xtab"] = xtab_frame

    # print("Success!")
    
with pd.ExcelWriter("reports/preanalysis_report.xlsx", engine="openpyxl") as writer:
    for report in preanalysis_report.keys():
        preanalysis_report[report].to_excel(writer, sheet_name=report)


# FACTOR ANALYSIS LOOP 

# Defining configuration dicts 
FA_SAMPLE_MASKS_DICT = {}

for config in FA_SAMPLE_CONFIG_DICT.keys():
    if FA_SAMPLE_CONFIG_DICT[config]["filter_mask"] != "nan":
        FA_SAMPLE_MASKS_DICT[f"{config}"] = clean_data.eval(FA_SAMPLE_CONFIG_DICT[config]["filter_mask"])
    else:
        FA_SAMPLE_MASKS_DICT[f"{config}"] = pd.Series(True, index=clean_data.index)

FA_CONFIG_GRIDS_DICT = {}

for config in FA_CONFIG_FRAME.columns:
    FA_CONFIG_GRIDS_DICT[f"{config}"] = FA_CONFIG_FRAME[config].dropna().index.to_list()

# Main loop
loading_matrices, accumulated_variances, updated_data, grid_KMO_scores, attribute_KMO_scores = factor_analysis_generation(
    data_frame=clean_data,
    sample_filters=FA_SAMPLE_MASKS_DICT,
    grids=FA_CONFIG_GRIDS_DICT,
    att_labels=LABELS_DICT
)

# Output
with pd.ExcelWriter('reports/factor_analysis_report.xlsx') as writer:
    pd.Series(accumulated_variances).to_excel(writer, sheet_name='Explained Variance')
    pd.Series(grid_KMO_scores).to_excel(writer, sheet_name='Grid KMO Daignostics')
    pd.Series(attribute_KMO_scores).to_excel(writer, sheet_name='Variables KMO Diagnostics')
    for excercise in loading_matrices.keys():
        loading_matrices[excercise].to_excel(writer, sheet_name=excercise)

# Outputting the filtered datasets for ABSA / SmartPLS
output_dir_data = Path("./filtered_data")
output_dir_data.mkdir(parents=True, exist_ok=True)

for sample in FA_SAMPLE_MASKS_DICT.keys():
    output_filtered_frame = clean_data[FA_SAMPLE_MASKS_DICT[sample]]
    output_filtered_frame.to_csv(output_dir_data/f"data_{sample}.csv", index=False)